# 🇧🇩 Bangla Mini-GPT — train on a free T4

**Before you start:** `Runtime → Change runtime type → T4 GPU`, and upload `bangla-mini-gpt-code.zip` to **My Drive/bangla-mini-gpt/**.

Everything important (tokenizer, token files, checkpoints, the final `.gguf`) is saved to Drive, so a disconnect costs nothing — just run all cells again.

In [ ]:
SIZE = 'base'          # 'small' (~17M, ~40 min) or 'base' (~49M, ~2-3 h over sessions)
MAX_MINUTES = 170      # save and stop before Colab cuts the session

from google.colab import drive
drive.mount('/content/drive')
WORK = '/content/drive/MyDrive/bangla-mini-gpt'
!mkdir -p /content/bmg && cd /content/bmg && unzip -oq {WORK}/bangla-mini-gpt-code.zip
%cd /content/bmg
!pip -q install sentencepiece
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1 · Data + tokenizer  (first run ~15 min, later runs copy from Drive in seconds)

In [ ]:
import os
cache = f'{WORK}/cache'
if os.path.exists(f'{cache}/tokens/train.bin'):
    print('using cached tokenizer + tokens from Drive')
    !mkdir -p data tokenizer && cp -r {cache}/tokens data/ && cp {cache}/bn_sp.* tokenizer/
else:
    !python prepare_data.py
    !python train_tokenizer.py --vocab 16000
    !python tokenize_data.py
    !mkdir -p {cache} && cp -r data/tokens {cache}/ && cp tokenizer/bn_sp.* {cache}/
!cat data/tokens/meta.json

## 2 · Train
Watch `loss` fall and the ✍ samples turn from noise into Bangla. If Colab disconnects, re-run everything — `--resume` continues from the last checkpoint.

If `--compile` errors on your runtime, delete it (training is just slower).

In [ ]:
!python train.py --size {SIZE} --out {WORK}/out/{SIZE} --resume --compile --max-minutes {MAX_MINUTES}

## 3 · Loss curve

In [ ]:
import json, matplotlib.pyplot as plt
rows = [json.loads(l) for l in open(f'{WORK}/out/{SIZE}/log.jsonl', encoding='utf-8')]
plt.figure(figsize=(8, 4))
plt.plot([r['step'] for r in rows], [r['train'] for r in rows], label='train')
plt.plot([r['step'] for r in rows], [r['val'] for r in rows], label='val')
plt.xlabel('step'); plt.ylabel('loss'); plt.legend(); plt.grid(alpha=.3); plt.title(f'Bangla Mini-GPT ({SIZE})')
plt.show()
print('latest sample:', rows[-1]['sample'])

## 4 · Let it write

In [ ]:
!python sample.py --ckpt {WORK}/out/{SIZE}/ckpt.pt --prompt 'বাংলাদেশের রাজধানী' --n 3 --temperature 0.7

## 5 · Export to GGUF for Ollama
Download the folder **My Drive/bangla-mini-gpt/export/** to `bangla-mini-gpt/export/` on your PC, then:
```
cd export\base
ollama create bangla-mini-gpt-base -f Modelfile
ollama run bangla-mini-gpt-base "বাংলাদেশ"
```

In [ ]:
!python export.py --size {SIZE} --ckpt {WORK}/out/{SIZE}/ckpt.pt --dest {WORK}/export/{SIZE}
!ls -lh {WORK}/export/{SIZE}